# Production ETL — Full 2025 Batch

Independent raw CSV → quality gate → processed v2 → curated v2. This notebook overwrites v2 paths if run again.

In [ ]:

from pyspark.sql import functions as F

# Flight Delay Prediction — Production ETL
storage = "stflight60106354"

raw_path = (
    f"abfss://raw@{storage}.dfs.core.windows.net/"
    "flight_delays/2025/csv/"
)

processed_path_v2 = (
    f"abfss://processed@{storage}.dfs.core.windows.net/"
    "flight_delays/2025/v2/clean_parquet/"
)

curated_path_v2 = (
    f"abfss://curated@{storage}.dfs.core.windows.net/"
    "flight_delays/2025/v2/model_ready/"
)

# 1. Check source files
csv_files = [
    f for f in dbutils.fs.ls(raw_path)
    if f.name.lower().endswith(".csv")
]

assert len(csv_files) == 12, "Expected 12 monthly CSV files"

# 2. Read raw dataset
raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "FAILFAST")
    .csv(raw_path)
)

required = [
    "Year", "Month", "DayofMonth", "FlightDate",
    "Reporting_Airline", "Origin", "Dest",
    "CRSDepTime", "Cancelled", "Diverted",
    "ArrDelay", "ArrDel15"
]

missing = [c for c in required if c not in raw.columns]
assert not missing, f"Missing columns: {missing}"

# 3. Convert fields
checked = (
    raw
    .withColumn("YearNum", F.col("Year").cast("int"))
    .withColumn("MonthNum", F.col("Month").cast("int"))
    .withColumn("FlightDateParsed", F.to_date("FlightDate"))
    .withColumn("CancelledNum", F.col("Cancelled").cast("double"))
    .withColumn("DivertedNum", F.col("Diverted").cast("double"))
    .withColumn("ArrDelayNum", F.col("ArrDelay").cast("double"))
    .withColumn("TargetNum", F.col("ArrDel15").cast("double"))
    .withColumn("CRSDepNum", F.col("CRSDepTime").cast("int"))
)

valid_time = (
    (
        F.col("CRSDepNum").between(0, 2359)
        & ((F.col("CRSDepNum") % 100) < 60)
    )
    | (F.col("CRSDepNum") == 2400)
)

# 4. Apply data quality rules
valid = (
    (F.col("YearNum") == 2025)
    & F.col("MonthNum").between(1, 12)
    & F.col("FlightDateParsed").isNotNull()
    & (F.year("FlightDateParsed") == F.col("YearNum"))
    & (F.month("FlightDateParsed") == F.col("MonthNum"))
    & (F.dayofmonth("FlightDateParsed") == F.col("DayofMonth").cast("int"))
    & F.col("Reporting_Airline").isNotNull()
    & (F.trim("Reporting_Airline") != "")
    & F.col("Origin").isNotNull()
    & (F.trim("Origin") != "")
    & F.col("Dest").isNotNull()
    & (F.trim("Dest") != "")
    & valid_time
    & (F.col("CancelledNum") == 0)
    & (F.col("DivertedNum") == 0)
    & F.col("ArrDelayNum").isNotNull()
    & F.col("TargetNum").isin(0.0, 1.0)
    & (
        F.col("TargetNum") ==
        F.when(F.col("ArrDelayNum") >= 15, 1.0).otherwise(0.0)
    )
)

checked = checked.withColumn(
    "QualityStatus",
    F.when(valid, "ACCEPTED").otherwise("REJECTED")
)

counts = {
    r["QualityStatus"]: r["count"]
    for r in checked.groupBy("QualityStatus").count().collect()
}

total = sum(counts.values())
accepted = counts.get("ACCEPTED", 0)
rejected = counts.get("REJECTED", 0)

assert total == accepted + rejected

print("Total:", total)
print("Accepted:", accepted)
print("Rejected:", rejected)

# 5. Quality gate — must pass before publishing
if total <= 0 or accepted / total < 0.90:
    raise ValueError("QUALITY FAILURE: Publication blocked")

print(f"Quality gate PASSED: {accepted / total:.2%}")

# 6. Create processed data
clean = (
    checked.filter(F.col("QualityStatus") == "ACCEPTED")
    .select(
        F.col("FlightDateParsed").alias("FlightDate"),
        F.col("YearNum").alias("Year"),
        F.col("MonthNum").alias("Month"),
        F.trim("Reporting_Airline").alias("Reporting_Airline"),
        F.trim("Origin").alias("Origin"),
        F.trim("Dest").alias("Dest"),
        F.col("CRSDepNum").alias("CRSDepTime"),
        F.col("ArrDelayNum").alias("ArrDelay"),
        F.col("TargetNum").cast("int").alias("ArrDel15")
    )
    .withColumn(
        "DepHour",
        F.when(F.col("CRSDepTime") == 2400, 0)
         .otherwise(F.floor(F.col("CRSDepTime") / 100))
         .cast("int")
    )
    .withColumn("DayOfWeek", F.dayofweek("FlightDate"))
    .withColumn(
        "IsWeekend",
        F.when(F.col("DayOfWeek").isin(1, 7), 1).otherwise(0)
    )
    .withColumn("Quarter", F.quarter("FlightDate"))
    .withColumn("Route", F.concat_ws("-", "Origin", "Dest"))
)

# 7. Publish processed Parquet
clean.write.mode("overwrite").format("parquet") \
    .partitionBy("Month").save(processed_path_v2)

# 8. Create curated model-ready data
model_columns = [
    "FlightDate", "Year", "Month",
    "Reporting_Airline", "Origin", "Dest",
    "CRSDepTime", "DepHour", "DayOfWeek",
    "IsWeekend", "Quarter", "Route", "ArrDel15"
]

model_ready = clean.select(*model_columns)

assert "ArrDelay" not in model_ready.columns

# 9. Publish curated Parquet
model_ready.write.mode("overwrite").format("parquet") \
    .partitionBy("Month").save(curated_path_v2)

# 10. Verify both published datasets
processed_count = spark.read.parquet(processed_path_v2).count()
curated_count = spark.read.parquet(curated_path_v2).count()

assert processed_count == accepted
assert curated_count == accepted

print("PASS: Production ETL completed")
print("Processed records:", processed_count)
print("Curated records:", curated_count)
print("PASS: Raw-to-curated pipeline verified")
